# 05 - Statistical Analysis, Hypothesis Testing & Thesis Publication Figures

This notebook brings together the **Statistical Hypothesis Testing Engine** and the **Publication Figure & Storyboard Generator** into a unified scientific analysis pipeline.

### 🔬 Part I: Statistical Hypothesis Testing & Reporting
1. **Omnibus Kruskal-Wallis Test**: Multi-group non-parametric difference test across all solvers per problem condition.
2. **Pairwise Mann-Whitney U Tests with FDR**: Two-sided comparisons with **Benjamini-Hochberg False Discovery Rate** correction ($\alpha = 0.05$).
3. **Effect Size Estimation**: Vargha-Delaney $\hat{A}_{12}$ stochastic dominance metric.
4. **Synthesis Transfer Correlation**: Pearson $r, p$ connecting synthesis fitness to empirical benchmark error.
5. **Master Markdown Report**: Automated scientific summary exported to `results/reports/comprehensive_master_report.md`.

---
### 📊 Part II: Thesis Publication Figures & Visual Storyboard
- **Figure E**: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension).
- **Figure 1 (RQ1)**: Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
- **Figure 2 (RQ2/RQ3)**: Empirical Convergence Trajectories & Target Precision ECDFs with IQR shaded bounds.
- **Figure 3 (RQ3 Hero)**: Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Figure 4 (RQ2/3 Ablation)**: Prompt Scaffolding Ablation across LLM model families.

In [1]:
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from collections import defaultdict
import colorsys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from shared.config import RESULTS_DIR
from shared.database import create_db_session_factory
from benchmarking.infra.storage import SQLiteSynthesisReadRepository
from benchmarking.infra.io.trace_repository import IOHTraceReader
from benchmarking.application.statistical_service import StatisticalEvaluationService
from benchmarking.domain.enums import BBOBFunction
from benchmarking.domain import EvaluationCondition, EvaluationDataset, RunTrace
from benchmarking.domain.services.resolvers import (
    resolve_canonical_model_slug,
    resolve_folder_solver_name,
)
from benchmarking.domain.services.palette import (
    FONT_FAMILY,
    SOLVER_PALETTE,
    SOLVER_LINE_STYLES,
    STRATEGY_PALETTE,
    MODEL_SCALE_PALETTE,
    REGIME_PALETTE,
    DIMENSION_PALETTE_CLEAN,
    DIMENSION_PALETTE_NOISY,
    get_solver_color,
    get_solver_line_style,
    get_rgba_fill,
    get_dimension_color,
    build_dynamic_solver_palette,
)

session_factory = create_db_session_factory()
sqlite_repo = SQLiteSynthesisReadRepository(session_factory)
trace_reader = IOHTraceReader()
service = StatisticalEvaluationService(sqlite_repo=sqlite_repo, trace_repo=trace_reader)

# ── 1. Thematic Publication Subdirectories ─────────────────────────────────
PUBLICATION_DIR   = RESULTS_DIR / "publication"
MAIN_RESULTS_DIR  = PUBLICATION_DIR / "main_results"
ABLATION_DIR      = PUBLICATION_DIR / "ablation"
EFFECT_SIZES_DIR  = PUBLICATION_DIR / "effect_sizes"
NOISE_DIR         = PUBLICATION_DIR / "noise_robustness"
CONVERGENCE_DIR   = PUBLICATION_DIR / "convergence"
PROFILES_DIR      = RESULTS_DIR / "figures" / "profiles"
STATISTICS_DIR    = RESULTS_DIR / "statistics"

for d in [MAIN_RESULTS_DIR, ABLATION_DIR, EFFECT_SIZES_DIR, NOISE_DIR, CONVERGENCE_DIR, PROFILES_DIR, STATISTICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPORTS_DIR       = RESULTS_DIR / "reports"
EVALUATIONS_DIR   = RESULTS_DIR / "ioh_traces"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None

print("✅ Statistical service and unified dynamic thesis visualization palette initialized.")

✅ Statistical service and unified dynamic thesis visualization palette initialized.


## Part I: Statistical Hypothesis Testing & Reporting
Ingest empirical benchmark traces and conduct rigorous non-parametric hypothesis testing with FDR control.

In [2]:
# Ingest benchmark traces and synthesis records
df_exp, df_iter = service.get_synthesis_dataframes()
all_benchmark_data = service.load_evaluation_traces(
    dims=FILTER_DIMS,
    problems=FILTER_PROBLEMS,
    noise_stds=FILTER_NOISE_STDS,
    solver_resolver=resolve_folder_solver_name,
)

if not all_benchmark_data:
    raise RuntimeError(f'No benchmark traces found in {EVALUATIONS_DIR}!')

all_dims = all_benchmark_data.dims
all_noise_stds = all_benchmark_data.noise_stds
clean_std = 0.0 if 0.0 in all_noise_stds else (all_noise_stds[0] if all_noise_stds else 0.0)
noisy_std = next((n for n in all_noise_stds if n > 0.0), all_noise_stds[-1] if all_noise_stds else 0.05)
PROBLEM_IDS = all_benchmark_data.problem_ids

DISCOVERED_SOLVERS = all_benchmark_data.solvers
SOLVER_PALETTE = build_dynamic_solver_palette(DISCOVERED_SOLVERS)

MODELS_TO_SOLVERS = defaultdict(list)
for s in DISCOVERED_SOLVERS:
    if ' / ' in s:
        MODELS_TO_SOLVERS[s.split(' / ')[0]].append(s)

LLM_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' in s]
CLASSICAL_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' not in s]
ALL_SOLVERS_ORDER = LLM_SOLVERS_ORDER + CLASSICAL_SOLVERS_ORDER

print(f'📦 Loaded {len(df_exp)} experiments and {len(all_benchmark_data)} problem conditions.')
print(f'🎯 Problems: {PROBLEM_IDS} | Dimensions: {all_dims} | Solvers: {DISCOVERED_SOLVERS}')


📦 Loaded 412 experiments and 30 problem conditions.
🎯 Problems: [1, 8, 11, 15, 21] | Dimensions: [2, 3, 5] | Solvers: ['CMA-ES', 'DE', 'PSO', 'Qwen2.5-Coder-14B / baseline', 'Qwen2.5-Coder-14B / guided', 'Qwen2.5-Coder-14B / thinking', 'Qwen2.5-Coder-14B / vectorization', 'Qwen2.5-Coder-7B / baseline', 'Qwen2.5-Coder-7B / guided', 'Qwen2.5-Coder-7B / thinking', 'Qwen2.5-Coder-7B / vectorization']


In [3]:
# ── 1. Omnibus Kruskal-Wallis & Pairwise FDR Tests ─────────────────────────
df_omnibus = service.run_omnibus_kruskal(all_benchmark_data)
df_pairwise = service.run_pairwise_fdr(all_benchmark_data, alpha=0.05)
r_val, p_val = service.compute_synthesis_transfer_correlation(df_exp)

print(f'✅ Omnibus Tests: {len(df_omnibus)} rows ({len(df_omnibus[df_omnibus["Significant"] == "Yes"])} significant)')
print(f'✅ Pairwise FDR Tests: {len(df_pairwise)} rows ({len(df_pairwise[df_pairwise["Significant (FDR)"]])} significant)')
print(f'✅ Synthesis Transfer Correlation: r = {r_val:.3f} (p = {p_val:.3e})')

# Export Master Markdown Report
report_path = REPORTS_DIR / 'comprehensive_master_report.md'
service.generate_markdown_report(df_omnibus=df_omnibus, df_pairwise=df_pairwise, df_exp=df_exp, output_path=report_path)
print(f'🎉 Master Comprehensive Report generated: {report_path}')


✅ Omnibus Tests: 30 rows (30 significant)
✅ Pairwise FDR Tests: 1630 rows (1277 significant)
✅ Synthesis Transfer Correlation: r = 0.000 (p = 1.000e+00)
🎉 Master Comprehensive Report generated: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/reports/comprehensive_master_report.md


## Part II: Thesis Publication Figures & Visual Storyboard
Render high-DPI thesis figures and storyboard artifacts.

# ── Figure E: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension) ──
clean_std = 0.0
noisy_std = 0.05

for dim in all_dims:


In [4]:
# ── Figure 5: Landscape Fragility Matrix (Clean → Noisy Degradation) ─────────
for dim in all_dims:
    frag_matrix, p_labels = service.compute_fragility_matrix(
        all_benchmark_data, dim, ALL_SOLVERS_ORDER, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig5 = go.Figure(data=go.Heatmap(
        z=frag_matrix,
        x=ALL_SOLVERS_ORDER,
        y=p_labels,
        colorscale="RdBu",
        zmid=0,
        colorbar=dict(title="<b>Fragility Δ</b>", thickness=12, len=0.85)
    ))
    fig5.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 5: Landscape Fragility Matrix (Clean → Noisy Degradation) — {dim}D</b><br><sup>Performance Drop (Δ = Clean Success Rate - Noisy Success Rate) Across Solvers by BBOB Problem</sup>",
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
            x=0.02, y=0.96
        ),
        width=1150, height=540,
        margin=dict(l=180, r=40, t=110, b=120),
        xaxis=dict(title="<b>Optimization Solver</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickangle=-30, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B")),
        yaxis=dict(title="<b>BBOB Problem Function</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"))
    )
    out_p = NOISE_DIR / f"fig_05_noise_fragility_matrix_{dim}D.png"
    fig5.write_image(str(out_p), scale=3)

print("✅ Figure 5 (Fragility Matrix) generated in results/publication/noise_robustness/")


2026-08-29 01:03:16 INFO Chromium init'ed with kwargs {}
2026-08-29 01:03:16 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:03:16 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpedl14zj5.
2026-08-29 01:03:16 INFO Opening browser.
2026-08-29 01:03:16 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpl173y7pe.
2026-08-29 01:03:16 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpl173y7pe
2026-08-29 01:03:18 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpedl14zj5/index.html
2026-08-29 01:03:19 INFO Getting tab from queue (has 1)
2026-08-29 01:03:19 INFO Got 55ED
2026-08-29 01:03:19 INFO Reloading tab 55ED before return.
2026-08-29 01:03:19 INFO Putting tab 55ED back (queue size: 0).
2026-08-29 01:03:19 INFO Waiting for all cleanups to finish.
2026-08-29 01:03:19 INFO Exiting Kaleido.
2026-08-29 01:03:19 INFO T

✅ Figure 5 (Fragility Matrix) generated in results/publication/noise_robustness/


### 📊 Model-Specific Hardness Success Rates (Clean vs. Noisy)
Separates the mean success rate analysis per LLM model (, ) across clean and noisy landscapes.

In [5]:
# ── Model-Specific Success Rate by Landscape Hardness (Clean vs. Noisy) ──
def render_model_success_rate_by_hardness(model_tag: str, solvers_list: list, dim: int):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"<b>(A) Deterministic Landscape (σ={clean_std}, {dim}D)</b>",
            f"<b>(B) Noisy Stochastic Landscape (σ={noisy_std}, {dim}D)</b>"
        ),
        horizontal_spacing=0.10
    )
    
    for c_idx, noise_level in enumerate([clean_std, noisy_std], start=1):
        df_hard = service.compute_hardness_success_rates(all_benchmark_data, dim, solvers_list, noise_level=noise_level)
        for solver in solvers_list:
            sub_s = df_hard[df_hard["Solver"] == solver]
            if not sub_s.empty:
                is_classical = " / " not in solver
                fig.add_trace(
                    go.Bar(
                        name=solver,
                        x=sub_s["Class"],
                        y=sub_s["Success Rate"],
                        marker=dict(
                            color=get_solver_color(solver),
                            line=dict(color="#0F172A", width=0.8)
                        ),
                        showlegend=(c_idx == 1)
                    ),
                    row=1, col=c_idx
                )
                
    fig.update_xaxes(tickangle=-15, tickfont=dict(size=10, family=FONT_FAMILY), row=1, col=1)
    fig.update_xaxes(tickangle=-15, tickfont=dict(size=10, family=FONT_FAMILY), row=1, col=2)
    fig.update_yaxes(title="<b>Target Success Rate (Δy ≤ 10⁻⁸)</b>", range=[0, 1.08], showgrid=True, gridcolor="#F1F5F9", row=1, col=1)
    fig.update_yaxes(range=[0, 1.08], showgrid=True, gridcolor="#F1F5F9", row=1, col=2)
    
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Empirical Success Rate by BBOB Landscape Hardness — {model_tag.upper()} ({dim}D)</b><br><sup>Comparison of Target Precision Hitting Rates Across 5 Problem Classes in Deterministic vs. Noisy Regimes</sup>",
            x=0.02, y=0.96,
            font=dict(size=14, color="#1E293B", family=FONT_FAMILY)
        ),
        barmode="group",
        bargap=0.25,
        bargroupgap=0.08,
        width=1220, height=560,
        margin=dict(l=70, r=40, t=100, b=120),
        legend=dict(
            orientation="h",
            yanchor="top", y=-0.22,
            xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="#E2E8F0",
            borderwidth=1,
            font=dict(size=11, family=FONT_FAMILY)
        )
    )
    
    slug = resolve_canonical_model_slug(model_tag)
    m_dir = PROFILES_DIR / slug / f"{dim}D"
    m_dir.mkdir(parents=True, exist_ok=True)
    out_p = m_dir / "figure_success_rate_by_hardness.png"
    fig.write_image(str(out_p), scale=3)

for dim in all_dims:
    for model_name, solvers_list in MODELS_TO_SOLVERS.items():
        solvers_to_plot = solvers_list + CLASSICAL_SOLVERS_ORDER
        render_model_success_rate_by_hardness(model_name, solvers_to_plot, dim)

print("✅ Model-specific success rate by hardness generated for all models and dimensions.")


2026-08-29 01:03:24 INFO TemporaryDirectory.cleanup() worked.
2026-08-29 01:03:24 INFO shutil.rmtree worked.
2026-08-29 01:03:24 INFO Chromium init'ed with kwargs {}
2026-08-29 01:03:24 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:03:24 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpt_w2pr9l.
2026-08-29 01:03:24 INFO Opening browser.
2026-08-29 01:03:24 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpk1h53x7y.
2026-08-29 01:03:24 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpk1h53x7y
2026-08-29 01:03:26 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpt_w2pr9l/index.html
2026-08-29 01:03:27 INFO Getting tab from queue (has 1)
2026-08-29 01:03:27 INFO Got 70AF
2026-08-29 01:03:27 INFO Reloading tab 70AF before return.
2026-08-29 01:03:27 INFO Putting tab 70AF back (queue size: 0).
2026-08-29 01:03:27 

✅ Model-specific success rate by hardness generated for all models and dimensions.


# 🎓 Part II: Thesis Visual Storyboard (RQ1 → RQ2 → RQ3 → Scaffolding Narrative Chain)

The following four figures form the core visual evidence for the thesis, saved into `results/figures/{dim}D/thesis/`:
- **Figure 1 (RQ1):** Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
- **Figure 2 (RQ2):** LLaMEA Synthesis Competency vs. Classical Baselines (Clean Convergence Trajectories & IQR).
- **Figure 3 (RQ3 Hero):** Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Figure 4 (RQ2/3 Ablation):** Prompt Scaffolding Ablation on LLaMEA-14B (Baseline vs. Guided vs. Thinking vs. Vectorization).


In [6]:
# ── THESIS Figure 1: Benchmark Difficulty under Noise Extension ──────────────
for dim in all_dims:
    clean_meds, noisy_meds, p_labels = service.compute_validation_medians(
        all_benchmark_data, dim, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        name=f"Deterministic (σ={clean_std})",
        x=p_labels,
        y=np.maximum(clean_meds, 1e-16),
        marker=dict(color="#1E3A8A", line=dict(color="#0F172A", width=1.2))
    ))
    fig1.add_trace(go.Bar(
        name=f"Noisy Stochastic (σ={noisy_std})",
        x=p_labels,
        y=np.maximum(noisy_meds, 1e-16),
        marker=dict(
            color="#EA580C",
            pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.4, size=8),
            line=dict(color="#7C2D12", width=1.2)
        )
    ))
    fig1.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 1: Benchmark Problem Difficulty under Stochastic Noise — {dim}D</b><br><sup>Median Terminal Optimization Error (Δy) Across All Solvers by BBOB Landscape Class</sup>",
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
            x=0.02, y=0.96
        ),
        xaxis=dict(title="<b>BBOB Landscape Class</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B")),
        yaxis=dict(
            type="log", title="<b>Median Final Error log₁₀(Δy)</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"), range=[-16, 4],
            showgrid=True, gridwidth=1, gridcolor="#F1F5F9"
        ),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1120, height=600,
        margin=dict(l=85, r=40, t=110, b=110),
        legend=dict(
            orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
            font=dict(size=14, family=FONT_FAMILY)
        )
    )
    out_p = MAIN_RESULTS_DIR / f"fig_01_benchmark_difficulty_{dim}D.png"
    fig1.write_image(str(out_p), scale=3)

print("✅ Figure 1 (Benchmark Difficulty) generated in results/publication/main_results/")


2026-08-29 01:03:37 INFO TemporaryDirectory.cleanup() worked.
2026-08-29 01:03:37 INFO shutil.rmtree worked.
2026-08-29 01:03:37 INFO Chromium init'ed with kwargs {}
2026-08-29 01:03:37 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:03:37 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpmmkebhl1.
2026-08-29 01:03:37 INFO Opening browser.
2026-08-29 01:03:37 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp7dtsae90.
2026-08-29 01:03:37 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp7dtsae90
2026-08-29 01:03:38 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpmmkebhl1/index.html
2026-08-29 01:03:39 INFO Getting tab from queue (has 1)
2026-08-29 01:03:39 INFO Got DF2D
2026-08-29 01:03:39 INFO Reloading tab DF2D before return.
2026-08-29 01:03:39 INFO Putting tab DF2D back (queue size: 0).
2026-08-29 01:03:39 

✅ Figure 1 (Benchmark Difficulty) generated in results/publication/main_results/


In [7]:
# ── THESIS: Multi-Panel Model Convergence & Empirical Runtime ECDFs ─────────
targets = np.logspace(-8, 2, 51)
eval_grid = np.logspace(0, 5, 200)

for dim in all_dims:
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        slug = resolve_canonical_model_slug(model_name)
        m_dir = PROFILES_DIR / slug / f"{dim}D"
        m_dir.mkdir(parents=True, exist_ok=True)
        solvers_to_plot = s_list + CLASSICAL_SOLVERS_ORDER
        for n_std, label_env in [(clean_std, "Deterministic (σ=0.0)"), (noisy_std, "Noisy Stochastic (σ=0.05)")]:
            env_dir = m_dir / f"std_{n_std}"
            env_dir.mkdir(parents=True, exist_ok=True)
            coords = [((i // 3) + 1, (i % 3) + 1) for i in range(len(PROBLEM_IDS) + 1)]
            subplot_titles = [f"<b>{BBOBFunction.get_name(p).replace(' Multi-Modal', '')}</b>" for p in PROBLEM_IDS] + ["<b>Overall Aggregate Summary</b>"]
            
            # 1. Multi-panel Trajectory Convergence (5 Problems + Overall Mean)
            fig_m = make_subplots(
                rows=2, cols=3,
                subplot_titles=subplot_titles,
                vertical_spacing=0.18,
                horizontal_spacing=0.08
            )
            for idx, p_id in enumerate(PROBLEM_IDS):
                r_idx, c_idx = coords[idx]
                cond = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                s_dict = all_benchmark_data.get(cond, {})
                for s_name in solvers_to_plot:
                    runs = s_dict.get(s_name, [])
                    if runs:
                        med, q25, q75, _ = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        color = get_solver_color(s_name)
                        s_style = get_solver_line_style(s_name)
                        is_classical = " / " not in s_name
                        fig_m.add_trace(go.Scatter(
                            x=eval_grid, y=med, mode="lines", name=s_name,
                            line=dict(color=s_style["color"], width=s_style["width"], dash=s_style["dash"]),
                            showlegend=(idx == 0)
                        ), row=r_idx, col=c_idx)
                
                fig_m.update_xaxes(type="log", title_text="<b>Evaluations</b>", title_font=dict(size=11, family=FONT_FAMILY), tickfont=dict(size=10, family=FONT_FAMILY), showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx)
                p_max_exp = 8 if p_id == 11 else (6 if p_id == 8 else 4)
                fig_m.update_yaxes(type="log", title_text="<b>Error Δy</b>", title_font=dict(size=11, family=FONT_FAMILY), tickfont=dict(size=10, family=FONT_FAMILY), range=[-16, p_max_exp], showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx)
            
            # Panel 6 for fig_m: Overall Convergence (Mean of problem medians)
            r_idx6, c_idx6 = coords[len(PROBLEM_IDS)]
            for s_name in solvers_to_plot:
                all_p_meds = []
                for p_id in PROBLEM_IDS:
                    cond = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                    runs = all_benchmark_data.get(cond, {}).get(s_name, [])
                    if runs:
                        med, _, _, _ = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        all_p_meds.append(med)
                if all_p_meds:
                    mean_med = np.nanmean(all_p_meds, axis=0)
                    color = get_solver_color(s_name)
                    s_style = get_solver_line_style(s_name)
                    is_classical = " / " not in s_name
                    fig_m.add_trace(go.Scatter(
                        x=eval_grid, y=mean_med, mode="lines", name=s_name,
                        line=dict(color=s_style["color"], width=s_style["width"], dash=s_style["dash"]),
                        showlegend=False
                    ), row=r_idx6, col=c_idx6)
            fig_m.update_xaxes(type="log", title_text="<b>Evaluations</b>", title_font=dict(size=11, family=FONT_FAMILY), tickfont=dict(size=10, family=FONT_FAMILY), showgrid=True, gridcolor="#F1F5F9", row=r_idx6, col=c_idx6)
            fig_m.update_yaxes(type="log", title_text="<b>Mean Error Δy</b>", title_font=dict(size=11, family=FONT_FAMILY), tickfont=dict(size=10, family=FONT_FAMILY), range=[-16, 8], showgrid=True, gridcolor="#F1F5F9", row=r_idx6, col=c_idx6)
            
            for anno in fig_m.layout.annotations:
                anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

            fig_m.update_layout(
                template="plotly_white",
                title=dict(
                    text=f"<b>Empirical Convergence Trajectories [{label_env}] — {model_name} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Log-scale Median Convergence Across 5 BBOB Problem Classes vs. Classical Baselines</span>",
                    font=dict(size=16, color="#0F172A", family=FONT_FAMILY),
                    x=0.02, y=0.97
                ),
                width=1320, height=840,
                margin=dict(l=70, r=40, t=110, b=120),
                legend=dict(
                    orientation="h",
                    yanchor="top", y=-0.14,
                    xanchor="center", x=0.5,
                    bgcolor="rgba(255,255,255,0.95)",
                    bordercolor="#E2E8F0",
                    borderwidth=1,
                    font=dict(size=12, family=FONT_FAMILY)
                )
            )
            out_m_traj = env_dir / "convergence_trajectories.png"
            fig_m.write_image(str(out_m_traj), scale=3)

            # 2. Multi-panel Empirical Runtime ECDF (5 Problems + Overall Aggregate)
            fig_ecdf = make_subplots(
                rows=2, cols=3,
                subplot_titles=subplot_titles,
                vertical_spacing=0.18,
                horizontal_spacing=0.08
            )
            for idx, p_id in enumerate(PROBLEM_IDS):
                r_idx, c_idx = coords[idx]
                cond = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                s_dict = all_benchmark_data.get(cond, {})
                for s_name in solvers_to_plot:
                    runs = s_dict.get(s_name, [])
                    if runs:
                        _, _, _, ecdf_curve = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        color = get_solver_color(s_name)
                        s_style = get_solver_line_style(s_name)
                        is_classical = " / " not in s_name
                        fig_ecdf.add_trace(go.Scatter(
                            x=eval_grid, y=ecdf_curve, mode="lines", name=s_name,
                            line=dict(color=s_style["color"], width=s_style["width"], dash=s_style["dash"]),
                            showlegend=(idx == 0)
                        ), row=r_idx, col=c_idx)
                fig_ecdf.update_xaxes(type="log", title_text="<b>Evaluations</b>", title_font=dict(size=11, family=FONT_FAMILY), tickfont=dict(size=10, family=FONT_FAMILY), showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx)
                fig_ecdf.update_yaxes(title_text="<b>Proportion Solved</b>", title_font=dict(size=11, family=FONT_FAMILY), tickfont=dict(size=10, family=FONT_FAMILY), range=[-0.02, 1.05], showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx)
            
            # Panel 6: Overall Aggregate Runtime ECDF (Mean across all problems)
            r_idx6, c_idx6 = coords[len(PROBLEM_IDS)]
            for s_name in solvers_to_plot:
                all_problem_ecdfs = []
                for p_id in PROBLEM_IDS:
                    cond = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                    runs = all_benchmark_data.get(cond, {}).get(s_name, [])
                    if runs:
                        _, _, _, ecdf_curve = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        all_problem_ecdfs.append(ecdf_curve)
                if all_problem_ecdfs:
                    mean_ecdf = np.mean(all_problem_ecdfs, axis=0)
                    color = get_solver_color(s_name)
                    s_style = get_solver_line_style(s_name)
                    is_classical = " / " not in s_name
                    fig_ecdf.add_trace(go.Scatter(
                        x=eval_grid, y=mean_ecdf, mode="lines", name=s_name,
                        line=dict(color=s_style["color"], width=s_style["width"], dash=s_style["dash"]),
                        showlegend=False
                    ), row=r_idx6, col=c_idx6)
            fig_ecdf.update_xaxes(type="log", title_text="<b>Evaluations</b>", title_font=dict(size=11, family=FONT_FAMILY), tickfont=dict(size=10, family=FONT_FAMILY), showgrid=True, gridcolor="#F1F5F9", row=r_idx6, col=c_idx6)
            fig_ecdf.update_yaxes(title_text="<b>Overall Proportion</b>", title_font=dict(size=11, family=FONT_FAMILY), tickfont=dict(size=10, family=FONT_FAMILY), range=[-0.02, 1.05], showgrid=True, gridcolor="#F1F5F9", row=r_idx6, col=c_idx6)
            
            for anno in fig_ecdf.layout.annotations:
                anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

            fig_ecdf.update_layout(
                template="plotly_white",
                title=dict(
                    text=f"<b>Empirical Runtime Cumulative Distribution Functions (ECDF) [{label_env}] — {model_name} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Proportion of Targets Solved (51 Logarithmic Targets in 10⁻⁸ ≤ Δy ≤ 10²) vs. Function Evaluation Budget Across 5 BBOB Problem Classes</span>",
                    font=dict(size=16, color="#0F172A", family=FONT_FAMILY),
                    x=0.02, y=0.97
                ),
                width=1320, height=840,
                margin=dict(l=70, r=40, t=110, b=120),
                legend=dict(
                    orientation="h",
                    yanchor="top", y=-0.14,
                    xanchor="center", x=0.5,
                    bgcolor="rgba(255,255,255,0.95)",
                    bordercolor="#E2E8F0",
                    borderwidth=1,
                    font=dict(size=12, family=FONT_FAMILY)
                )
            )
            out_m_ecdf = env_dir / "target_precision_ecdf.png"
            fig_ecdf.write_image(str(out_m_ecdf), scale=3)

print("✅ 6-Panel Convergence and BBOB Runtime ECDF profiles generated successfully.")



2026-08-29 01:03:44 INFO TemporaryDirectory.cleanup() worked.
2026-08-29 01:03:44 INFO shutil.rmtree worked.
2026-08-29 01:03:44 INFO TemporaryDirectory.cleanup() worked.
2026-08-29 01:03:44 INFO shutil.rmtree worked.
2026-08-29 01:03:44 INFO Chromium init'ed with kwargs {}
2026-08-29 01:03:44 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:03:44 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp46qiimeb.
2026-08-29 01:03:44 INFO Opening browser.
2026-08-29 01:03:44 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpntikq32v.
2026-08-29 01:03:44 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpntikq32v
2026-08-29 01:03:45 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp46qiimeb/index.html
2026-08-29 01:03:46 INFO Getting tab from queue (has 1)
2026-08-29 01:03:46 INFO Got D7DB
2026-08-29 01:03:46 INFO Reloading

✅ 6-Panel Convergence and BBOB Runtime ECDF profiles generated successfully.


In [8]:
# ── THESIS Figure 6: Cross-Environment Noise Robustness Profile ──────────────
for dim in all_dims:
    valid_solvers, clean_rates, noisy_rates, deltas = service.compute_robustness_profile(
        all_benchmark_data, dim, ALL_SOLVERS_ORDER, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig6 = go.Figure()
    fig6.add_trace(go.Bar(name=f"Deterministic (σ={clean_std})", x=valid_solvers, y=clean_rates, marker=dict(color="#1E3A8A", line=dict(color="#0F172A", width=1.2))))
    fig6.add_trace(go.Bar(name=f"Noisy Stochastic (σ={noisy_std})", x=valid_solvers, y=noisy_rates, marker=dict(color="#EA580C", pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8), line=dict(color="#7C2D12", width=1.2))))
    for s, c_r, n_r, delta in zip(valid_solvers, clean_rates, noisy_rates, deltas):
        drop_pct = (delta / c_r * 100) if c_r > 0 else 0.0
        fig6.add_annotation(x=s, y=max(c_r, n_r) + 0.04, text=f"<b>-Δ{drop_pct:.0f}%</b>" if delta > 0 else "<b>0%</b>", showarrow=False, font=dict(size=10, color="#EA580C" if drop_pct > 25 else "#16A34A", family=FONT_FAMILY))
    fig6.update_layout(
        template="plotly_white",
        title=dict(text=f"<b>Figure 6: Cross-Environment Noise Robustness Profile — {dim}D</b><br><sup>Generalization Retention: Clean vs. Noisy Target Success Rate (Δy ≤ 10⁻⁸) with Performance Drop Badges</sup>", font=dict(size=20, color="#0F172A", family=FONT_FAMILY), x=0.02, y=0.96),
        xaxis=dict(title="<b>Optimization Solver</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickangle=-30, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B")),
        yaxis=dict(title="<b>Overall Target Success Rate</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"), range=[0, 1.20], showgrid=True, gridcolor="#E2E8F0", gridwidth=1.2),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1220, height=660,
        margin=dict(l=75, r=40, t=100, b=120),
        legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5, bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1, font=dict(size=11, family=FONT_FAMILY))
    )
    out_p = NOISE_DIR / f"fig_06_robustness_profile_{dim}D.png"
    fig6.write_image(str(out_p), scale=3)

print("✅ Figure 6 (Robustness Profile) generated in results/publication/noise_robustness/")


2026-08-29 01:04:34 INFO TemporaryDirectory.cleanup() worked.
2026-08-29 01:04:34 INFO shutil.rmtree worked.
2026-08-29 01:04:34 INFO Chromium init'ed with kwargs {}
2026-08-29 01:04:34 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:04:34 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpzlh7ycct.
2026-08-29 01:04:34 INFO Opening browser.
2026-08-29 01:04:34 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmphddh36eb.
2026-08-29 01:04:34 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmphddh36eb
2026-08-29 01:04:35 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpzlh7ycct/index.html
2026-08-29 01:04:36 INFO Getting tab from queue (has 1)
2026-08-29 01:04:36 INFO Got 5173
2026-08-29 01:04:36 INFO Reloading tab 5173 before return.
2026-08-29 01:04:36 INFO Putting tab 5173 back (queue size: 0).
2026-08-29 01:04:36 

✅ Figure 6 (Robustness Profile) generated in results/publication/noise_robustness/


In [9]:
# ── THESIS Figure 3: Prompt Strategy & Model Scale Ablation ──────────────────
for dim in all_dims:
    fig3 = go.Figure()
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        strat_labels, clean_rates, noisy_rates = service.compute_scaffolding_ablation(
            all_benchmark_data, dim, s_list, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
        )
        is_14b = "14B" in model_name
        color_c = "#0284C7" if is_14b else "#7DD3FC"
        color_n = "#D97706" if is_14b else "#FCD34D"
        fig3.add_trace(go.Bar(name=f"{model_name} (Clean)", x=strat_labels, y=clean_rates, marker=dict(color=color_c, line=dict(color="#0F172A", width=1.0))))
        fig3.add_trace(go.Bar(name=f"{model_name} (Noisy)", x=strat_labels, y=noisy_rates, marker=dict(color=color_n, pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8), line=dict(color="#7C2D12", width=1.0))))
    fig3.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 3: Prompt Strategy & Model Scale Ablation — {dim}D</b><br><sup>Empirical Target Success Rate (Δy ≤ 10⁻⁸) by Prompt Scaffolding and Model Scale in Clean vs. Noisy Regimes</sup>",
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY), x=0.02, y=0.96
        ),
        xaxis=dict(title="<b>Prompt Scaffolding Strategy</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B")),
        yaxis=dict(title="<b>Target Success Rate</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"), range=[0, 1.15], showgrid=True, gridcolor="#E2E8F0", gridwidth=1.2),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1180, height=620,
        margin=dict(l=75, r=40, t=100, b=90),
        legend=dict(
            orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
            font=dict(size=11, family=FONT_FAMILY)
        )
    )
    out_p = ABLATION_DIR / f"fig_03_prompt_strategy_ablation_{dim}D.png"
    fig3.write_image(str(out_p), scale=3)

print("✅ Figure 3 (Prompt Strategy Ablation) generated in results/publication/ablation/")


2026-08-29 01:04:40 INFO Chromium init'ed with kwargs {}
2026-08-29 01:04:40 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:04:40 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpuph_2ubc.
2026-08-29 01:04:40 INFO Opening browser.
2026-08-29 01:04:40 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpbotqslvq.
2026-08-29 01:04:40 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpbotqslvq
2026-08-29 01:04:41 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpuph_2ubc/index.html
2026-08-29 01:04:42 INFO Getting tab from queue (has 1)
2026-08-29 01:04:42 INFO Got 5ECC
2026-08-29 01:04:42 INFO Reloading tab 5ECC before return.
2026-08-29 01:04:42 INFO Putting tab 5ECC back (queue size: 0).
2026-08-29 01:04:42 INFO Waiting for all cleanups to finish.
2026-08-29 01:04:42 INFO Exiting Kaleido.
2026-08-29 01:04:42 INFO T

✅ Figure 3 (Prompt Strategy Ablation) generated in results/publication/ablation/


In [10]:
# ── THESIS Figure 4: Vargha-Delaney Effect Size (A12) Heatmap ────────────────
a12_matrix = np.full((len(ALL_SOLVERS_ORDER), len(ALL_SOLVERS_ORDER)), 0.5)
for i, s1 in enumerate(ALL_SOLVERS_ORDER):
    for j, s2 in enumerate(ALL_SOLVERS_ORDER):
        if i == j:
            a12_matrix[i, j] = 0.5
        else:
            sub = df_pairwise[(df_pairwise["Solver 1"] == s1) & (df_pairwise["Solver 2"] == s2)]
            if not sub.empty:
                a12_matrix[i, j] = sub["A12"].mean()
            else:
                sub_rev = df_pairwise[(df_pairwise["Solver 1"] == s2) & (df_pairwise["Solver 2"] == s1)]
                if not sub_rev.empty:
                    a12_matrix[i, j] = 1.0 - sub_rev["A12"].mean()

fig4_a12 = go.Figure(data=go.Heatmap(
    z=a12_matrix, x=ALL_SOLVERS_ORDER, y=ALL_SOLVERS_ORDER, colorscale="RdBu_r", zmid=0.5, zmin=0.0, zmax=1.0,
    text=[[f"{val:.2f}" for val in row] for row in a12_matrix], texttemplate="%{text}", textfont=dict(size=10, family=FONT_FAMILY),
    colorbar=dict(title="<b>Â₁₂ Metric</b>", tickvals=[0.0, 0.29, 0.5, 0.71, 1.0], ticktext=["0.0 (Col Large)", "0.29 (Col Med)", "0.50 (Tie)", "0.71 (Row Med)", "1.0 (Row Large)"], thickness=14, len=0.85)
))
fig4_a12.update_layout(
    title=dict(text="<b>Figure 4: Global Vargha-Delaney Effect Size (Â₁₂) Heatmap</b><br><sup>Pairwise Non-Parametric Effect Sizes Averaged Across All 30 Problem Conditions (Row vs. Column)</sup>", font=dict(size=20, color="#0F172A", family=FONT_FAMILY), x=0.02, y=0.96),
    template="plotly_white", width=1050, height=880, margin=dict(l=180, r=40, t=95, b=140),
    xaxis=dict(title="<b>Comparison Solver</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickangle=-35, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B")), yaxis=dict(title="<b>Reference Solver</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"), autorange="reversed")
)
out_fig4 = EFFECT_SIZES_DIR / "fig_04_a12_heatmap.png"
fig4_a12.write_image(str(out_fig4), scale=3)
print("✅ Figure 4 (A12 Heatmap) generated in results/publication/effect_sizes/")


2026-08-29 01:04:46 INFO Chromium init'ed with kwargs {}
2026-08-29 01:04:46 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:04:46 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpfdnyy95y.
2026-08-29 01:04:46 INFO Opening browser.
2026-08-29 01:04:46 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5d0uj7__.
2026-08-29 01:04:46 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5d0uj7__
2026-08-29 01:04:47 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpfdnyy95y/index.html
2026-08-29 01:04:48 INFO Getting tab from queue (has 1)
2026-08-29 01:04:48 INFO Got 3B0D
2026-08-29 01:04:48 INFO Reloading tab 3B0D before return.
2026-08-29 01:04:48 INFO Putting tab 3B0D back (queue size: 0).
2026-08-29 01:04:48 INFO Waiting for all cleanups to finish.
2026-08-29 01:04:48 INFO Exiting Kaleido.
2026-08-29 01:04:48 INFO T

✅ Figure 4 (A12 Heatmap) generated in results/publication/effect_sizes/


In [11]:
# ── THESIS Figure 7: Pairwise Win / Tie / Loss Summary Ranking ────────────────
win_counts = df_pairwise[df_pairwise["Significant (FDR)"] == True]["Outcome"].value_counts()
win_data = []
for s in ALL_SOLVERS_ORDER:
    w_cnt = win_counts.get(f"{s} Wins", 0)
    win_data.append({"Solver": s, "FDR Wins": w_cnt, "Type": "Classical" if " / " not in s else "LLM Synthesized"})
df_wins = pd.DataFrame(win_data).sort_values(by="FDR Wins", ascending=True)

fig7 = go.Figure(go.Bar(
    x=df_wins["FDR Wins"], y=df_wins["Solver"], orientation="h",
    marker=dict(color=[get_solver_color(s) for s in df_wins["Solver"]], line=dict(color="#0F172A", width=1.0)),
    text=df_wins["FDR Wins"], textposition="outside", textfont=dict(size=11, family=FONT_FAMILY, color="#1E293B")
))
fig7.update_layout(
    template="plotly_white",
    title=dict(text="<b>Figure 7: Global Pairwise Win Summary (Mann-Whitney U with FDR Correction)</b><br><sup>Total Significant Head-to-Head Victories (p < 0.05) Across All 30 BBOB Problem Conditions (1,630 Total Comparisons)</sup>", font=dict(size=20, color="#0F172A", family=FONT_FAMILY), x=0.02, y=0.96),
    xaxis=dict(title="<b>Statistically Significant Wins (FDR Adjusted)</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"), range=[0, 220], showgrid=True, gridcolor="#E2E8F0", gridwidth=1.2),
    yaxis=dict(title="<b>Optimization Solver</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B")),
    width=1050, height=600, margin=dict(l=220, r=40, t=95, b=65)
)
out_fig7 = MAIN_RESULTS_DIR / "fig_07_win_tie_loss.png"
fig7.write_image(str(out_fig7), scale=3)
print("✅ Figure 7 (Win/Tie/Loss Ranking) generated in results/publication/main_results/")


2026-08-29 01:04:48 INFO Chromium init'ed with kwargs {}
2026-08-29 01:04:48 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:04:48 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpd9y__0q6.
2026-08-29 01:04:48 INFO Opening browser.
2026-08-29 01:04:48 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpqz9q9kwp.
2026-08-29 01:04:48 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpqz9q9kwp
2026-08-29 01:04:49 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpd9y__0q6/index.html
2026-08-29 01:04:50 INFO Getting tab from queue (has 1)
2026-08-29 01:04:50 INFO Got 61F6
2026-08-29 01:04:50 INFO Reloading tab 61F6 before return.
2026-08-29 01:04:50 INFO Putting tab 61F6 back (queue size: 0).
2026-08-29 01:04:50 INFO Waiting for all cleanups to finish.
2026-08-29 01:04:50 INFO Exiting Kaleido.
2026-08-29 01:04:50 INFO T

✅ Figure 7 (Win/Tie/Loss Ranking) generated in results/publication/main_results/


In [12]:
# ── THESIS Figure 9B: Empirical Runtime ECDF Performance by Dimension ────────
df_all = service.compute_auc_ecdf_matrix(all_benchmark_data, ALL_SOLVERS_ORDER, eval_grid, targets, group_by="condition")
solver_overall_auc = df_all.groupby("Solver")["AUC-ECDF (%)"].mean().sort_values(ascending=True)
sorted_solvers = list(solver_overall_auc.index)

fig9b = make_subplots(
    rows=1, cols=2,
    subplot_titles=["<b>(A) Clean (σ=0.0)</b>", "<b>(B) Noisy (σ=0.05)</b>"],
    horizontal_spacing=0.12,
    shared_yaxes=True
)

dims = [2, 3, 5]

for dim in dims:
    df_clean_dim = df_all[(df_all["Dim"] == dim) & (df_all["Noise Std"] == 0.0)]
    val_map_clean = df_clean_dim.groupby("Solver")["AUC-ECDF (%)"].mean().to_dict()
    y_vals = sorted_solvers
    x_vals_clean = [val_map_clean.get(s, 0.0) for s in y_vals]
    fig9b.add_trace(go.Bar(
        y=y_vals, x=x_vals_clean, orientation="h",
        name=f"{dim}D",
        marker=dict(color=get_dimension_color(dim, is_noisy=False), line=dict(color="#0F172A", width=0.8)),
        text=[f"{v:.1f}%" if v > 0 else "" for v in x_vals_clean],
        textposition="outside",
        textfont=dict(size=9, family=FONT_FAMILY, color="#1E293B"),
        showlegend=True,
    ), row=1, col=1)

for dim in dims:
    df_noisy_dim = df_all[(df_all["Dim"] == dim) & (df_all["Noise Std"] == 0.05)]
    val_map_noisy = df_noisy_dim.groupby("Solver")["AUC-ECDF (%)"].mean().to_dict()
    y_vals = sorted_solvers
    x_vals_noisy = [val_map_noisy.get(s, 0.0) for s in y_vals]
    fig9b.add_trace(go.Bar(
        y=y_vals, x=x_vals_noisy, orientation="h",
        name=f"{dim}D",
        marker=dict(
            color=get_dimension_color(dim, is_noisy=True),
            pattern=REGIME_PALETTE["noisy"]["pattern"],
            line=dict(color=REGIME_PALETTE["noisy"]["border"], width=0.8)
        ),
        text=[f"{v:.1f}%" if v > 0 else "" for v in x_vals_noisy],
        textposition="outside",
        textfont=dict(size=9, family=FONT_FAMILY, color="#1E293B"),
        showlegend=False,
    ), row=1, col=2)

fig9b.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9B: Empirical Runtime ECDF Performance by Dimension</b><br><sup>Area Under Runtime ECDF (AUC-ECDF %) Across 2D, 3D, and 5D Problems in Clean vs. Noisy Regimes</sup>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    barmode="group", bargap=0.20, bargroupgap=0.05,
    width=1350, height=720,
    margin=dict(l=220, r=60, t=110, b=80),
    legend=dict(
        title="<b>Dimension:</b>",
        orientation="h", yanchor="top", y=-0.08, xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
        font=dict(size=10, family=FONT_FAMILY)
    )
)
fig9b.update_xaxes(title_text="<b>AUC-ECDF (%)</b>", range=[0, 60], showgrid=True, gridcolor="#F1F5F9", row=1, col=1)
fig9b.update_xaxes(title_text="<b>AUC-ECDF (%)</b>", range=[0, 60], showgrid=True, gridcolor="#F1F5F9", row=1, col=2)
fig9b.update_yaxes(tickfont=dict(size=10, family=FONT_FAMILY), row=1, col=1)

out_9b = MAIN_RESULTS_DIR / "fig_09b_auc_ecdf_by_dim.png"
fig9b.write_image(str(out_9b), scale=3)
print("✅ Figure 9B (By Dimension) generated in results/publication/main_results/")


2026-08-29 01:04:50 INFO TemporaryDirectory.cleanup() worked.
2026-08-29 01:04:50 INFO shutil.rmtree worked.
2026-08-29 01:04:50 INFO Chromium init'ed with kwargs {}
2026-08-29 01:04:50 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:04:50 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmprgepirrn.
2026-08-29 01:04:50 INFO Opening browser.
2026-08-29 01:04:50 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpg6uv2c8l.
2026-08-29 01:04:50 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpg6uv2c8l
2026-08-29 01:04:51 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmprgepirrn/index.html
2026-08-29 01:04:51 INFO TemporaryDirectory.cleanup() worked.
2026-08-29 01:04:51 INFO shutil.rmtree worked.
2026-08-29 01:04:51 INFO TemporaryDirectory.cleanup() worked.
2026-08-29 01:04:51 INFO shutil.rmtree worked.
2026-08-29 01:0

✅ Figure 9B (By Dimension) generated in results/publication/main_results/


In [13]:
# ── THESIS Figure 9C: Cross-Environment Noise Robustness & Retention Profile ──
clean_solver_auc = df_all[df_all["Noise Std"] == 0.0].groupby("Solver")["AUC-ECDF (%)"].mean()
noisy_solver_auc = df_all[df_all["Noise Std"] == 0.05].groupby("Solver")["AUC-ECDF (%)"].mean()
df_ret = pd.DataFrame({"Clean AUC": clean_solver_auc, "Noisy AUC": noisy_solver_auc}).dropna()
df_ret["Retention Ratio (%)"] = (df_ret["Noisy AUC"] / (df_ret["Clean AUC"] + 1e-6)) * 100.0
df_ret = df_ret.sort_values(by="Clean AUC", ascending=True)

fig9c = make_subplots(
    rows=1, cols=2,
    subplot_titles=["<b>(A) Clean (σ=0.0)</b>", "<b>(B) Noisy (σ=0.05) & Retention</b>"],
    horizontal_spacing=0.12,
    shared_yaxes=True
)
fig9c.add_trace(go.Bar(
    y=df_ret.index, x=df_ret["Clean AUC"], orientation="h",
    name="Clean",
    marker=dict(color=[get_solver_color(s) for s in df_ret.index], line=dict(color="#0F172A", width=0.8)),
    text=[f"{v:.1f}%" for v in df_ret["Clean AUC"]],
    textposition="outside",
    textfont=dict(size=10, family=FONT_FAMILY, color="#1E293B"),
    showlegend=False
), row=1, col=1)

fig9c.add_trace(go.Bar(
    y=df_ret.index, x=df_ret["Noisy AUC"], orientation="h",
    name="Noisy",
    marker=dict(
        color="rgba(255,255,255,0.85)",
        pattern=REGIME_PALETTE["noisy"]["pattern"],
        line=dict(color=[get_solver_color(s) for s in df_ret.index], width=1.5)
    ),
    text=[f"{v:.1f}% (<b>{r:.0f}% ret.</b>)" for v, r in zip(df_ret["Noisy AUC"], df_ret["Retention Ratio (%)"])],
    textposition="outside",
    textfont=dict(size=10, family=FONT_FAMILY, color="#1E293B"),
    showlegend=False
), row=1, col=2)

fig9c.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9C: Cross-Environment Noise Robustness & Performance Retention Profile</b><br><sup>Empirical Target Progress (AUC-ECDF %) in Clean vs. Noisy Regimes with Retention Ratios (AUC_noisy / AUC_clean × 100%)</sup>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1350, height=680,
    margin=dict(l=220, r=80, t=110, b=70),
)
fig9c.update_xaxes(title_text="<b>Clean AUC-ECDF (%)</b>", range=[0, 50], showgrid=True, gridcolor="#F1F5F9", row=1, col=1)
fig9c.update_xaxes(title_text="<b>Noisy AUC-ECDF (%)</b>", range=[0, 50], showgrid=True, gridcolor="#F1F5F9", row=1, col=2)
fig9c.update_yaxes(tickfont=dict(size=10, family=FONT_FAMILY), row=1, col=1)

out_9c = MAIN_RESULTS_DIR / "fig_09c_auc_ecdf_clean_vs_noisy.png"
fig9c.write_image(str(out_9c), scale=3)
print("✅ Figure 9C (Clean vs Noisy Retention) generated in results/publication/main_results/")


2026-08-29 01:04:52 INFO Chromium init'ed with kwargs {}
2026-08-29 01:04:52 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:04:52 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpdzph717e.
2026-08-29 01:04:52 INFO Opening browser.
2026-08-29 01:04:52 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpdcf3f3d1.
2026-08-29 01:04:52 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpdcf3f3d1
2026-08-29 01:04:53 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpdzph717e/index.html
2026-08-29 01:04:54 INFO Getting tab from queue (has 1)
2026-08-29 01:04:54 INFO Got 1D8F
2026-08-29 01:04:54 INFO Reloading tab 1D8F before return.
2026-08-29 01:04:54 INFO Putting tab 1D8F back (queue size: 0).
2026-08-29 01:04:54 INFO Waiting for all cleanups to finish.
2026-08-29 01:04:54 INFO Exiting Kaleido.
2026-08-29 01:04:54 INFO T

✅ Figure 9C (Clean vs Noisy Retention) generated in results/publication/main_results/


In [14]:
# ── THESIS Figure 9D: Solver × Problem Function Matrix (Heatmap) ─────────────
prob_ids = [1, 8, 11, 15, 21]
prob_labels = [f"{BBOBFunction.get_name(p)} (f{p})" for p in prob_ids]
solvers_y = list(solver_overall_auc.sort_values(ascending=False).index)

matrix_clean = np.zeros((len(solvers_y), len(prob_ids)))
matrix_noisy = np.zeros((len(solvers_y), len(prob_ids)))
for r_idx, s in enumerate(solvers_y):
    for c_idx, p in enumerate(prob_ids):
        c_sub = df_all[(df_all["Solver"] == s) & (df_all["Problem ID"] == p) & (df_all["Noise Std"] == 0.0)]
        n_sub = df_all[(df_all["Solver"] == s) & (df_all["Problem ID"] == p) & (df_all["Noise Std"] == 0.05)]
        matrix_clean[r_idx, c_idx] = c_sub["AUC-ECDF (%)"].mean() if not c_sub.empty else 0.0
        matrix_noisy[r_idx, c_idx] = n_sub["AUC-ECDF (%)"].mean() if not n_sub.empty else 0.0

fig9d = make_subplots(
    rows=1, cols=2,
    subplot_titles=["<b>(A) Clean (σ=0.0)</b>", "<b>(B) Noisy (σ=0.05)</b>"],
    horizontal_spacing=0.10,
    shared_yaxes=True
)
fig9d.add_trace(go.Heatmap(
    z=matrix_clean, x=prob_labels, y=solvers_y,
    colorscale="Blues", zmin=0, zmax=70,
    text=[[f"{v:.1f}%" for v in row] for row in matrix_clean],
    texttemplate="%{text}",
    textfont=dict(size=10, family=FONT_FAMILY),
    showscale=False,
), row=1, col=1)
fig9d.add_trace(go.Heatmap(
    z=matrix_noisy, x=prob_labels, y=solvers_y,
    colorscale="Blues", zmin=0, zmax=70,
    text=[[f"{v:.1f}%" for v in row] for row in matrix_noisy],
    texttemplate="%{text}",
    textfont=dict(size=10, family=FONT_FAMILY),
    colorbar=dict(
        title="<b>AUC-ECDF (%)</b>",
        title_side="top",
        tickfont=dict(size=10, family=FONT_FAMILY),
        len=0.85
    ),
), row=1, col=2)

fig9d.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9D: Solver Performance Matrix Across BBOB Problem Landscapes</b><br><sup>Area Under Runtime ECDF (AUC-ECDF %) Across Canonical Function Classes in Clean vs. Noisy Regimes</sup>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1400, height=680,
    margin=dict(l=220, r=40, t=110, b=90),
)
fig9d.update_xaxes(tickangle=-25, tickfont=dict(size=10, family=FONT_FAMILY), row=1, col=1)
fig9d.update_xaxes(tickangle=-25, tickfont=dict(size=10, family=FONT_FAMILY), row=1, col=2)
fig9d.update_yaxes(tickfont=dict(size=10, family=FONT_FAMILY), autorange="reversed", row=1, col=1)

out_9d = MAIN_RESULTS_DIR / "fig_09d_auc_ecdf_by_problem.png"
fig9d.write_image(str(out_9d), scale=3)
print("✅ Figure 9D (By Problem Heatmap) generated in results/publication/main_results/")


2026-08-29 01:04:54 INFO Chromium init'ed with kwargs {}
2026-08-29 01:04:54 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:04:54 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpo2x8ahz7.
2026-08-29 01:04:54 INFO Opening browser.
2026-08-29 01:04:54 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpzzumer0o.
2026-08-29 01:04:54 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpzzumer0o
2026-08-29 01:04:55 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpo2x8ahz7/index.html
2026-08-29 01:04:56 INFO Getting tab from queue (has 1)
2026-08-29 01:04:56 INFO Got 5D5A
2026-08-29 01:04:56 INFO Reloading tab 5D5A before return.
2026-08-29 01:04:56 INFO Putting tab 5D5A back (queue size: 0).
2026-08-29 01:04:56 INFO Waiting for all cleanups to finish.
2026-08-29 01:04:56 INFO Exiting Kaleido.
2026-08-29 01:04:56 INFO T

✅ Figure 9D (By Problem Heatmap) generated in results/publication/main_results/


In [15]:
# ── THESIS Figure 9E: LLM Parameter Scale Ablation (7B vs. 14B) Across Dims ──
fig9e = make_subplots(
    rows=1, cols=2,
    subplot_titles=["<b>(A) Clean (σ=0.0)</b>", "<b>(B) Noisy (σ=0.05)</b>"],
    horizontal_spacing=0.10,
    shared_yaxes=True
)
dims = [2, 3, 5]
sub_7b = df_all[df_all["Solver"].str.contains("7B")]
sub_14b = df_all[df_all["Solver"].str.contains("14B")]

for col_idx, (n_std, title_sfx) in enumerate([(0.0, "Clean"), (0.05, "Noisy")], start=1):
    c_7b = sub_7b[sub_7b["Noise Std"] == n_std]
    c_14b = sub_14b[sub_14b["Noise Std"] == n_std]
    vals_7b = [c_7b[c_7b["Dim"] == d]["AUC-ECDF (%)"].mean() for d in dims]
    vals_14b = [c_14b[c_14b["Dim"] == d]["AUC-ECDF (%)"].mean() for d in dims]
    
    fig9e.add_trace(go.Bar(
        x=[f"{d}D" for d in dims], y=vals_7b,
        name="Qwen2.5-Coder-7B",
        marker=dict(color=MODEL_SCALE_PALETTE["Qwen2.5-Coder-7B"], line=dict(color="#0F172A", width=0.8)),
        text=[f"{v:.1f}%" for v in vals_7b], textposition="outside",
        textfont=dict(size=9, family=FONT_FAMILY, color="#1E293B"),
        showlegend=(col_idx == 1)
    ), row=1, col=col_idx)
    
    fig9e.add_trace(go.Bar(
        x=[f"{d}D" for d in dims], y=vals_14b,
        name="Qwen2.5-Coder-14B",
        marker=dict(color=MODEL_SCALE_PALETTE["Qwen2.5-Coder-14B"], line=dict(color="#0F172A", width=0.8)),
        text=[f"{v:.1f}%" for v in vals_14b], textposition="outside",
        textfont=dict(size=9, family=FONT_FAMILY, color="#1E293B"),
        showlegend=(col_idx == 1)
    ), row=1, col=col_idx)
    
    for baseline in ["CMA-ES", "PSO", "DE"]:
        b_sub = df_all[(df_all["Solver"] == baseline) & (df_all["Noise Std"] == n_std)]
        b_vals = [b_sub[b_sub["Dim"] == d]["AUC-ECDF (%)"].mean() for d in dims]
        b_style = get_solver_line_style(baseline)
        fig9e.add_trace(go.Scatter(
            x=[f"{d}D" for d in dims], y=b_vals,
            mode="lines+markers", name=baseline,
            line=dict(color=b_style["color"], dash=b_style["dash"], width=2.2),
            marker=dict(size=8, symbol="diamond" if baseline=="CMA-ES" else ("square" if baseline=="PSO" else "circle")),
            showlegend=(col_idx == 1)
        ), row=1, col=col_idx)

fig9e.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9E: LLM Parameter Scale Ablation (7B vs. 14B) Across Dimensions</b><br><sup>Mean Area Under Runtime ECDF (AUC-ECDF %) for 7B vs. 14B Synthesized Optimizers vs. Classical Baselines in Clean and Noisy Regimes</sup>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    barmode="group",
    width=1350, height=620,
    margin=dict(l=70, r=40, t=110, b=90),
    legend=dict(
        orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
        font=dict(size=10, family=FONT_FAMILY)
    )
)
fig9e.update_yaxes(title_text="<b>Mean AUC-ECDF (%)</b>", range=[0, 60], showgrid=True, gridcolor="#F1F5F9", row=1, col=1)
fig9e.update_yaxes(range=[0, 60], showgrid=True, gridcolor="#F1F5F9", row=1, col=2)
fig9e.update_xaxes(title_text="<b>Problem Dimension</b>", row=1, col=1)
fig9e.update_xaxes(title_text="<b>Problem Dimension</b>", row=1, col=2)

out_9e = MAIN_RESULTS_DIR / "fig_09e_auc_ecdf_model_scale.png"
fig9e.write_image(str(out_9e), scale=3)
print("✅ Figure 9E (Model Scale Ablation) generated in results/publication/main_results/")


2026-08-29 01:04:57 INFO Chromium init'ed with kwargs {}
2026-08-29 01:04:57 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-29 01:04:57 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpp5nluymq.
2026-08-29 01:04:57 INFO Opening browser.
2026-08-29 01:04:57 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6leyluj4.
2026-08-29 01:04:57 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6leyluj4
2026-08-29 01:04:58 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpp5nluymq/index.html
2026-08-29 01:04:58 INFO Getting tab from queue (has 1)
2026-08-29 01:04:58 INFO Got 7E8E
2026-08-29 01:04:58 INFO Reloading tab 7E8E before return.
2026-08-29 01:04:58 INFO Putting tab 7E8E back (queue size: 0).
2026-08-29 01:04:58 INFO Waiting for all cleanups to finish.
2026-08-29 01:04:58 INFO Exiting Kaleido.
2026-08-29 01:04:58 INFO T

✅ Figure 9E (Model Scale Ablation) generated in results/publication/main_results/
